# Chapter 07 — Pandas II: Cleaning, Transformation, and Aggregation

Real datasets are rarely analysis-ready. Learn missing-value handling, duplicate detection, strings, dates, grouping, joins, and reshaping.

## Prerequisites
Chapter 6 DataFrame selection, dtypes, and inspection.

## Learning goals
- Diagnose missing values and duplicates.
- Use string, datetime, and categorical accessors.
- Transform with vectorized operations, map, and apply.
- Group, aggregate, merge, concatenate, pivot, and melt.

**Study sequence:** use the theory notebook first, then tutorial, then lab. The examples use the bundled synthetic data unless otherwise noted.

In [1]:
from pathlib import Path
ROOT=Path.cwd()
if not (ROOT/"data").exists() and (ROOT.parent/"data").exists(): ROOT=ROOT.parent
DATA=ROOT/"data"
print("Data directory:", DATA.resolve())

Data directory: /mnt/data/python_data_analysis_book/data


## Guided learning path

Work from top to bottom. Each step states the purpose, explains the API choice, and provides a complete code cell. Modify parameters and predict the output before running it.

### Step 1 — Guided 1 — Preserve raw inputs

Create a copy before changing values so transformations can be reviewed and compared.

In [2]:
from pathlib import Path
ROOT=Path.cwd()
if not (ROOT/"data").exists() and (ROOT.parent/"data").exists(): ROOT=ROOT.parent
DATA=ROOT/"data"
print("Data directory:", DATA.resolve())
import pandas as pd
raw=pd.read_csv(DATA/"messy_sales.csv",parse_dates=["order_date"]); clean=raw.copy()
print("raw:",raw.shape,"missing:",raw.isna().sum().to_dict())

Data directory: /mnt/data/python_data_analysis_book/data
raw: (62, 11) missing: {'order_id': 0, 'order_date': 0, 'region': 7, 'product': 0, 'units': 6, 'unit_price': 0, 'discount': 0, 'revenue': 0, 'cost': 0, 'profit': 0, 'customer_id': 0}


### Step 2 — Guided 2 — Standardize and handle nulls

Use explicit categories for unknown text rather than guessing the actual value. Median imputation is only a demonstration rule here.

In [3]:
clean["product"]=clean["product"].astype("string").str.strip().str.title()
clean["region"]=clean.region.astype("string").str.strip().str.title().fillna("Unknown")
clean["units"]=clean.units.fillna(clean.units.median())
clean=clean.drop_duplicates(subset="order_id",keep="first")
print(clean.shape,clean.isna().sum().to_dict())

(60, 11) {'order_id': 0, 'order_date': 0, 'region': 0, 'product': 0, 'units': 0, 'unit_price': 0, 'discount': 0, 'revenue': 0, 'cost': 0, 'profit': 0, 'customer_id': 0}


### Step 3 — Guided 3 — Aggregate and rank

`sort_values` is applied after aggregation, so each row represents a group.

In [4]:
result=(clean.groupby("region",as_index=False).agg(revenue=("revenue","sum"),profit=("profit","sum"),orders=("order_id","count")).sort_values("revenue",ascending=False))
display(result)

    region     revenue      profit  orders
4     West  4629593.20  1309589.11      15
2    South  1378468.64   254695.85      15
0     East  1369294.76   323101.85      16
3  Unknown  1146964.55   398004.35       6
1    North   256483.17    49049.87       8


### Step 4 — Guided 4 — Validate a customer join

A many-to-one join expects each customer ID to appear once in the lookup table. `indicator=True` shows match status.

In [5]:
orders=pd.read_csv(DATA/"orders.csv"); customers=pd.read_csv(DATA/"customers.csv")
joined=orders.merge(customers,on="customer_id",how="left",validate="many_to_one",indicator=True)
print("orders:",len(orders),"joined:",len(joined),"unmatched:",int((joined._merge!="both").sum()))

orders: 240 joined: 240 unmatched: 0


## Tutorial checkpoint

1. What was the input data type?
2. What did the function or method return?
3. Which argument most changed the result?
4. What edge case should be tested before using this in production?